# AutoPBR v2 — `run_pipeline.ipynb` (repo-friendly)

Notebook orchestratore per il repo `dera8/AutoPBR` (root scripts):

- `1_download_data.py` (opzionale)
- `2_sam3hi.py`
- `3_proxy_semantic_check.py`
- `4_nemotron_building_priors.py`
- `5_nemotron_fullimage.py` (baseline opzionale)

Esegue gli script **via CLI** (come terminale), quindi log live e riproducibilità.


In [ ]:
from pathlib import Path
import os, sys, subprocess, shlex

REPO_ROOT = Path(".").resolve()

# Se in futuro sposti gli script in scripts/, cambia qui:
SCRIPTS_DIR = REPO_ROOT

# Cartelle attese dal README del repo
PHOTOS_DIR = REPO_ROOT / "photos2"          # input images

print("Repo:", REPO_ROOT)
print("Scripts dir:", SCRIPTS_DIR)
print("Photos dir:", PHOTOS_DIR)


## Helper: run comandi con output live


In [ ]:
def run(cmd: str):
    print("\n▶", cmd)
    subprocess.run(shlex.split(cmd), check=True)

def py(script: str, args: str = ""):
    script_path = (SCRIPTS_DIR / script).resolve()
    if not script_path.exists():
        raise FileNotFoundError(f"Script not found: {script_path}")
    run(f"{sys.executable} {script_path} {args}".strip())


## Step 0 — Sanity checks (cartelle + API key)


In [ ]:
PHOTOS_DIR.mkdir(exist_ok=True, parents=True)

imgs = list(PHOTOS_DIR.glob("*.jpg")) + list(PHOTOS_DIR.glob("*.png")) + list(PHOTOS_DIR.glob("*.jpeg"))
print(f"Immagini trovate in {PHOTOS_DIR}: {len(imgs)}")
if len(imgs) == 0:
    print("⚠️ Metti le immagini dentro photos2/ prima di fare Run All.")

if not os.environ.get("NVIDIA_API_KEY"):
    print("⚠️ NVIDIA_API_KEY non impostata. Setta da terminale prima di aprire Jupyter, oppure qui:")
    print('   os.environ["NVIDIA_API_KEY"] = "..."')
else:
    print("✅ NVIDIA_API_KEY OK")


## Step 1 (opzionale) — Download data

Scommenta se vuoi scaricare nuovi dati (bbox/profile ecc.).
Se hai già i file e le foto, puoi saltarlo.


In [ ]:
# ESEMPIO:
# py("1_download_data.py", '--bbox "5.32045,60.39670,5.32650,60.39990" --profile autopbr')

print("Step 1 opzionale: scommenta la riga se ti serve.")


## Step 2 — Semantic segmentation (SAM3)


In [ ]:
py("2_sam3hi.py")


## Step 3 — Proxy semantic check


In [ ]:
py("3_proxy_semantic_check.py")


## Step 4 — Nemotron hierarchical (building priors)


In [ ]:
py("4_nemotron_building_priors.py")


## Step 5 (opzionale) — Baseline full-image


In [ ]:
# py("5_nemotron_fullimage.py")
print("Step 5 opzionale: scommenta per eseguire la baseline full-image.")


## Output quick-check


In [ ]:
expected = [
    REPO_ROOT / "materials_hybrid.json",
    REPO_ROOT / "baseline_full_image.json",
    REPO_ROOT / "materials_per_image_v2.json",
    REPO_ROOT / "materials_v2_filtered.json",
]

for p in expected:
    print(("✅" if p.exists() else "❌"), p)
